# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published on: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` field (Croissant schema). The list of record sets, fields, and columns is presented below.

In [ ]:
# List available record sets and their fields/columns
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []

if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} - Name: {rs.get('name', 'N/A')}")
        # List fields
        fields = rs.get('field', [])
        if not fields:
            print("  No fields found.")
        else:
            for field in fields:
                print(f"  Field @id: {field['@id']} - Name: {field.get('name', 'N/A')} - DataType: {field.get('dataType', 'N/A')}")
        # List columns (if present)
        columns = rs.get('column', [])
        if columns:
            for col in columns:
                print(f"  Column @id: {col['@id']} - Name: {col.get('name', 'N/A')} - DataType: {col.get('dataType', 'N/A')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets
dataframes = {}

if not record_sets:
    print("No record sets to extract data from.")
else:
    for rs in record_sets:
        rs_id = rs['@id']
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id} with shape: {df.shape}")

    # Example: show columns and preview for first record set
    main_record_set_id = record_sets[0]['@id'] if record_sets else None
    if main_record_set_id:
        print("Columns in main RecordSet:")
        print(dataframes[main_record_set_id].columns.tolist())
        display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.
Each field and group is referenced by its `@id` as defined in the Croissant schema.

In [ ]:
import numpy as np

# Select the main record set and numeric field based on the data overview
main_record_set_id = record_sets[0]['@id'] if record_sets else None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Identify a numeric field for analysis, for this dataset, GAD-7 score is typical
    numeric_field_id = 'GAD7_score'  # Example field name; replace with the actual @id if available
    group_field_id = 'village'       # Example group field (categorical); replace with actual @id as needed

    # If the field exists, continue with EDA
    if numeric_field_id in df.columns:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized" ]].head())

        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field '{numeric_field_id}' not found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot histogram of GAD7 scores and group means by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=15, kde=True, ax=ax, color='skyblue')
    ax.set_title(f'Histogram of {numeric_field_id}')
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel('Count')
    plt.show()

    # Grouped mean plot
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        fig, ax = plt.subplots(figsize=(10, 5))
        group_means.sort_values(ascending=False).plot(kind='bar', ax=ax)
        ax.set_title(f'Mean {numeric_field_id} by {group_field_id}')
        ax.set_xlabel(group_field_id)
        ax.set_ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County using the `mlcroissant` library.
- Identified main record sets and fields using their Croissant schema `@id`.
- Applied basic data filtering, normalization, and grouping using survey score fields.
- Visualized the distribution and village-level means of GAD7 scores, highlighting variation in mental health indicators across regions.
- The dataset is structured and AI-ready, supporting reproducible, scalable analyses for mental health and community research.